# 31 — Defenses and Privacy in Federated Learning — Lab

## Description

This lab extends the Flower/TensorFlow FedAvg simulation from m29 and the gradient-scaling attack from m30. You implement **Multi-Krum** (with Krum recovered as its $m=1$ special case) and Coordinate-wise Trimmed Mean as a *custom Flower aggregation strategy*, and run them, through real `fl.simulation.start_simulation()` rounds on MNIST, against the exact m30 attack: one or more malicious clients scaling their update by $\Lambda$. You sweep Multi-Krum's $m$ parameter to chart its performance-robustness trade-off. You then work through a numeric demonstration of **SecAgg**'s double-masking protocol, and implement **Local Differential Privacy** as a Flower client that clips and noises its own update before returning it, measuring the accuracy cost of a given privacy budget $\epsilon$ across real federated training runs.

## Objectives

- State the Krum aggregation rule and the client-count condition $n \geq 2f+3$ under which its Byzantine fault tolerance guarantee holds.
- Implement Multi-Krum (with Krum recovered as its $m=1$ special case) as a Flower aggregation strategy and explain the performance-robustness trade-off controlled by $m$.
- Explain the Coordinate-wise Trimmed Mean algorithm as an alternative robust aggregation rule.
- Describe the SecAgg double-masking protocol and state exactly what information it hides from the server.
- Apply Local Differential Privacy to client update vectors using the Gaussian mechanism and measure the accuracy cost relative to the privacy budget $\epsilon$.
- Reason about the combined effect of layering robust aggregation, SecAgg, and LDP.

## Prerequisites

You must have completed the **m29** (Federated Learning Fundamentals) and **m30** (Attacks on Federated Learning) lectures and labs. You should have a working Flower FedAvg simulation with 10 clients, MNIST data, a small Keras CNN, and server-side accuracy tracking across rounds, and you should understand the m30 gradient-scaling attack (a malicious client submits $\Lambda \cdot g_{\mathrm{mal}}$ instead of its honest update $g_{\mathrm{mal}}$). You should also have read the **m31 lecture** covering Krum, Multi-Krum, Coordinate-wise Trimmed Mean, SecAgg, and the Gaussian mechanism for Local Differential Privacy.

## Required Software/Packages

- Flower (flwr) >= 1.5.0, < 2.0 for the federated learning framework and simulation engine.
- TensorFlow >= 2.12.0 for the Keras CNN model.
- NumPy >= 1.23.0 for the aggregation-rule math.
- Matplotlib >= 3.7.0 for plotting.

## Environment/Setup Instructions

Run the following cells to install Flower (if needed) and import all required libraries.

**What this cell does:** Makes sure the Flower framework (`flwr`) is available, exactly as in m29/m30. It tries to import it and, if missing, installs it with pip, including the `[simulation]` extra (the virtual-client simulation engine used throughout this lab) with the `<2.0` pin that keeps the API used here stable.

In [ ]:
import subprocess
import sys

# Install Flower if not present
try:
    import flwr
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "flwr[simulation]>=1.5.0,<2.0"])
    import flwr
    print("Flower installed successfully.")

**What this cell does:** Imports NumPy, Matplotlib, TensorFlow/Keras, and Flower, then fixes the random seed. It loads MNIST and normalizes it exactly as in m29, then keeps a **smaller slice** (3,000 training images, 1,000 test images) before partitioning across clients — this lab reruns many federated simulations (a 4-way aggregation comparison, an $m$-sweep, and an $\epsilon$-sweep), so a smaller per-client dataset keeps the total notebook runtime reasonable. Server-side evaluation each round still uses the **full** 10,000-image MNIST test set, exactly as in m29, so the reported accuracy numbers remain meaningful.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import flwr as fl
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

# Load MNIST (28x28 grayscale, add a channel dimension), exactly as in m29.
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train_full = x_train_full[..., np.newaxis] / 255.0
x_test = x_test[..., np.newaxis] / 255.0
y_train_full, y_test = y_train_full.flatten(), y_test.flatten()

# A smaller slice for per-client training data, to keep the many simulations in this lab fast.
TRAIN_SUBSET = 3000
TEST_SUBSET = 1000
x_train = x_train_full[:TRAIN_SUBSET]
y_train = y_train_full[:TRAIN_SUBSET]
x_train_eval = x_train_full[:TEST_SUBSET]   # small client-side test slice (reusing train images' labels space)
y_train_eval = y_train_full[:TEST_SUBSET]

print(f"Per-client training pool: {x_train.shape}  |  Full server-side test set: {x_test.shape}")

## Background/Context

In m29, `fl.server.strategy.FedAvg` performed the weighted averaging internally — you never saw the list of individual client updates. Byzantine-robust aggregation needs exactly that list: Krum and Multi-Krum compute pairwise distances between whole client updates, and Trimmed Mean sorts each coordinate across clients. Flower exposes this through its `Strategy` interface: `aggregate_fit(self, server_round, results, failures)` receives the raw per-client `FitRes` objects (each with its uploaded parameters and sample count) and must return the aggregated parameters. The built-in `FedAvg` strategy's `aggregate_fit` computes a weighted mean; this lab instead subclasses `FedAvg` and overrides `aggregate_fit` to compute Krum, Multi-Krum, or Trimmed Mean on the same raw per-client updates.

Because these rules need one vector per client (not one array per layer), every client's list of weight arrays is flattened into a single 1-D vector before scoring or sorting, and the aggregated flat vector is reshaped back into per-layer arrays before being sent back to the clients — the `flatten_weights` / `unflatten_weights` helpers below do exactly that, once, so the rest of the code works with one vector per client regardless of how many layers the model has.

The malicious client reuses m30's exact attack, translated from PyTorch to Keras: compute the honest local update, then scale the *delta* from the received global weights by $\Lambda$ before returning it.

## Exercise Instructions

**Part A — Byzantine-Robust Aggregation as a Flower Strategy (25 min)**

1. Reuse the m29 CNN and `FedAvgClient`, and partition the smaller MNIST slice across 10 clients with `create_iid_partitions`. (3 min)
2. Implement `MaliciousClient`, which scales its weight delta by $\Lambda=50$ before returning it, exactly as in m30. (4 min)
3. Implement `flatten_weights`, `unflatten_weights`, `krum_scores`, `multi_krum` (with `krum` as its $m=1$ special case), and `trimmed_mean`. (6 min)
4. Implement `RobustFedAvg`, a `fl.server.strategy.FedAvg` subclass whose `aggregate_fit` dispatches to one of the four aggregation rules. (5 min)
5. Run real 8-round Flower simulations with 2 of 10 clients malicious ($\Lambda=50$), once per aggregation rule (FedAvg, Krum, Multi-Krum with $m=n-f=8$, Trimmed Mean with $\beta=f=2$). Plot per-round global accuracy for all four. (4 min)
6. Sweep Multi-Krum's $m$ over $\{2, 4, 6, 10\}$ under the same attack (reusing the $m=1$ and $m=8$ runs from step 5), and plot final-round accuracy against $m$. (3 min)

**Part B — Secure Aggregation (10 min)**

7. Implement the double-masking formula on 5 toy clients: pairwise masks that cancel exactly when summed, and a self-mask that does not. (5 min)
8. Implement a small Shamir secret-sharing example and confirm that $t$ shares reconstruct a secret exactly while $t-1$ shares are consistent with *any* candidate secret. (5 min)

**Part C — Local Differential Privacy as a Flower Client (15 min)**

9. Implement `PrivateClient`, which clips its weight delta to L2 norm $C$ and adds Gaussian noise calibrated to $(\epsilon, \delta)$ before returning it. (5 min)
10. Run real 8-round, all-honest Flower simulations for $\epsilon \in \{0.5, 1, 2, 5, 10\}$ plus a no-DP baseline, using plain `FedAvg` aggregation. Plot final-round accuracy against $\epsilon$. (10 min)

## Part A: Byzantine-Robust Aggregation as a Flower Strategy

**What this cell does:** Defines the same small CNN and `create_iid_partitions` function as m29 (unchanged), and the same `FedAvgClient` as m29's honest client. Partitions the 3,000-image training slice and 1,000-image test slice across `NUM_CLIENTS=10` clients.

In [ ]:
NUM_CLIENTS = 10

def create_model() -> keras.Model:
    '''Small CNN for MNIST, identical to m29.'''
    model = keras.Sequential([
        layers.Conv2D(16, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(32, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def create_iid_partitions(x_train, y_train, x_test, y_test, num_clients):
    '''Uniform IID partition across clients, identical to m29.'''
    train_idx = np.random.permutation(len(x_train))
    test_idx = np.random.permutation(len(x_test))
    train_splits = np.array_split(train_idx, num_clients)
    test_splits = np.array_split(test_idx, num_clients)
    return [
        (x_train[tr], y_train[tr], x_test[te], y_test[te])
        for tr, te in zip(train_splits, test_splits)
    ]

class FedAvgClient(fl.client.NumPyClient):
    '''Honest Flower client, identical to m29's.'''
    def __init__(self, x_train, y_train, x_test, y_test):
        self.x_train, self.y_train = x_train, y_train
        self.x_test, self.y_test = x_test, y_test
        self.model = create_model()

    def get_parameters(self, config: Dict) -> List[np.ndarray]:
        return self.model.get_weights()

    def fit(self, parameters: List[np.ndarray], config: Dict) -> Tuple[List[np.ndarray], int, Dict]:
        self.model.set_weights(parameters)
        local_epochs = config.get('local_epochs', 1)
        batch_size = config.get('batch_size', 32)
        self.model.fit(self.x_train, self.y_train, epochs=local_epochs, batch_size=batch_size, verbose=0)
        return self.model.get_weights(), len(self.x_train), {}

    def evaluate(self, parameters: List[np.ndarray], config: Dict) -> Tuple[float, int, Dict]:
        self.model.set_weights(parameters)
        loss, accuracy = self.model.evaluate(self.x_test, self.y_test, verbose=0)
        return loss, len(self.x_test), {"accuracy": accuracy}

partitions = create_iid_partitions(x_train, y_train, x_train_eval, y_train_eval, NUM_CLIENTS)
print(f"{NUM_CLIENTS} clients, {len(partitions[0][0])} training images each.")

**What this cell does:** Defines `MaliciousClient(FedAvgClient)`, which overrides `fit` to perform the identical m30 attack: train normally, then compute the *delta* between the received global weights and the newly trained weights, scale that delta by `scale_factor`, and return `initial + scale_factor * delta` instead of the honest result. `NUM_MALICIOUS=2` and `SCALE_FACTOR=50` match the m30 lab's attack strength; with $n=10$ and $f=2$, the Byzantine-tolerance condition $n \geq 2f+3=7$ holds.

In [ ]:
NUM_MALICIOUS = 2
SCALE_FACTOR = 50
assert NUM_CLIENTS >= 2 * NUM_MALICIOUS + 3, "Need n >= 2f+3 for Krum's guarantee to hold."

class MaliciousClient(FedAvgClient):
    '''Scales its weight delta by scale_factor before returning it -- the m30 attack.'''
    def __init__(self, x_train, y_train, x_test, y_test, scale_factor=SCALE_FACTOR):
        super().__init__(x_train, y_train, x_test, y_test)
        self.scale_factor = scale_factor

    def fit(self, parameters, config):
        initial_weights = [w.copy() for w in parameters]
        self.model.set_weights(parameters)
        local_epochs = config.get('local_epochs', 1)
        batch_size = config.get('batch_size', 32)
        self.model.fit(self.x_train, self.y_train, epochs=local_epochs, batch_size=batch_size, verbose=0)
        updated_weights = self.model.get_weights()
        scaled_weights = [
            init + self.scale_factor * (upd - init)
            for init, upd in zip(initial_weights, updated_weights)
        ]
        return scaled_weights, len(self.x_train), {}

print(f"NUM_MALICIOUS={NUM_MALICIOUS}, SCALE_FACTOR={SCALE_FACTOR}")
print(f"Byzantine-tolerance check: n >= 2f+3  ->  {NUM_CLIENTS} >= {2*NUM_MALICIOUS+3}  ->  "
      f"{NUM_CLIENTS >= 2*NUM_MALICIOUS+3}")

**The formulas.** Given $n$ client update vectors $v_1, \ldots, v_n$ (each client's full set of weight arrays, flattened into one vector) and an assumed bound $f$ on the number of Byzantine clients:

$$\mathrm{score}(i) = \sum_{j \in \mathcal{N}_i} \lVert v_i - v_j \rVert^2, \qquad \mathcal{N}_i = \text{the } (n-f-2) \text{ closest of the other } n-1 \text{ vectors to } v_i.$$

**Krum** keeps only the single lowest-scoring vector: $\mathrm{Krum}(v_1,\ldots,v_n) = v_{i^{*}}$, $i^{*}=\arg\min_i \mathrm{score}(i)$.

**Multi-Krum** generalizes this: sort all $n$ vectors by score and average the $m$ with the lowest scores:

$$\mathrm{MultiKrum}_m(v_1, \ldots, v_n) = \frac{1}{m}\sum_{i \in S} v_i, \qquad S = \text{the } m \text{ indices with the smallest score}(i).$$

$m=1$ recovers Krum exactly; $m=n$ recovers the plain FedAvg mean. The default, Byzantine-safe choice is $m=n-f$.

**Coordinate-wise Trimmed Mean** instead filters each coordinate $j$ independently:

$$\mathrm{TrimmedMean}_j(v_1,\ldots,v_n) = \frac{1}{n-2\beta}\sum_{i \,\in\, \mathrm{middle}(n-2\beta)} v_{i,j}.$$

**What this cell does:** `flatten_weights`/`unflatten_weights` convert between a Keras model's list of per-layer weight arrays and a single 1-D vector (and back), which is what lets the rest of this cell operate on "one vector per client" regardless of the model's architecture. `pairwise_sq_distances`, `krum_scores`, `multi_krum`, `krum`, and `trimmed_mean` implement the four formulas above exactly as in the lecture. Two assertions confirm the $m=1$ and $m=n$ special cases.

In [ ]:
def flatten_weights(ndarrays: List[np.ndarray]) -> Tuple[np.ndarray, List[Tuple[int, ...]]]:
    '''Flatten a list of per-layer weight arrays into one 1-D vector, plus their shapes.'''
    shapes = [a.shape for a in ndarrays]
    flat = np.concatenate([a.ravel() for a in ndarrays])
    return flat, shapes

def unflatten_weights(flat: np.ndarray, shapes: List[Tuple[int, ...]]) -> List[np.ndarray]:
    '''Inverse of flatten_weights: reshape a flat vector back into per-layer arrays.'''
    arrays, idx = [], 0
    for shape in shapes:
        size = int(np.prod(shape))
        arrays.append(flat[idx:idx + size].reshape(shape))
        idx += size
    return arrays

def pairwise_sq_distances(V: np.ndarray) -> np.ndarray:
    '''n x n matrix of squared Euclidean distances between the rows of V.'''
    diffs = V[:, None, :] - V[None, :, :]
    return np.sum(diffs ** 2, axis=-1)

def krum_scores(V: np.ndarray, f: int) -> np.ndarray:
    '''Krum score for every vector: sum of squared distances to its (n-f-2) nearest neighbors.'''
    n = len(V)
    k = n - f - 2
    assert k >= 1, "Need n >= f + 3 for Krum's score to be defined."
    D_mat = pairwise_sq_distances(V)
    scores = np.zeros(n)
    for i in range(n):
        dists_to_others = np.delete(D_mat[i], i)
        scores[i] = np.sort(dists_to_others)[:k].sum()
    return scores

def multi_krum(V: np.ndarray, f: int, m: int = None):
    '''Average the m lowest-scoring vectors. Defaults to m = n - f (the safe choice).'''
    n = len(V)
    if m is None:
        m = n - f
    scores = krum_scores(V, f)
    selected = np.argsort(scores)[:m]
    return V[selected].mean(axis=0), selected

def krum(V: np.ndarray, f: int):
    '''Krum is Multi-Krum with m=1 -- not a separate algorithm.'''
    return multi_krum(V, f, m=1)

def trimmed_mean(V: np.ndarray, beta: int) -> np.ndarray:
    '''Per-coordinate: drop the beta largest and beta smallest values, average the rest.'''
    n = len(V)
    assert 2 * beta < n, "Need 2*beta < n."
    S = np.sort(V, axis=0)
    return S[beta:n - beta].mean(axis=0)

# Sanity checks: the two named special cases of Multi-Krum.
_V_test = np.random.randn(NUM_CLIENTS, 50)
_agg_m1, _ = multi_krum(_V_test, NUM_MALICIOUS, m=1)
_agg_krum, _ = krum(_V_test, NUM_MALICIOUS)
assert np.allclose(_agg_m1, _agg_krum), "multi_krum(V, f, m=1) must equal krum(V, f)."
_agg_mn, _ = multi_krum(_V_test, NUM_MALICIOUS, m=NUM_CLIENTS)
assert np.allclose(_agg_mn, _V_test.mean(axis=0)), "multi_krum(V, f, m=n) must equal the plain mean."
print("PASS: multi_krum(m=1) == krum, and multi_krum(m=n) == plain mean.")

**What this cell does:** `RobustFedAvg` subclasses Flower's `FedAvg` strategy and overrides only `aggregate_fit`. Everything else (client sampling, `on_fit_config_fn`, server-side `evaluate_fn`) is inherited unchanged from `FedAvg`, exactly as in m29. Inside `aggregate_fit`: each client's returned `Parameters` is converted to a list of NumPy arrays (`fl.common.parameters_to_ndarrays`, the same call m29 already uses in its `evaluate_fn`), flattened to one vector per client, stacked into a matrix `V`, aggregated with whichever `aggregation_method` this strategy instance was constructed with, and the resulting flat vector is reshaped back into per-layer arrays and wrapped as `Parameters` for Flower to broadcast next round.

In [ ]:
class RobustFedAvg(fl.server.strategy.FedAvg):
    '''FedAvg with a pluggable aggregation rule: 'fedavg', 'krum', 'multikrum', or 'trimmedmean'.'''
    def __init__(self, aggregation_method="fedavg", f=0, m=None, **kwargs):
        super().__init__(**kwargs)
        self.aggregation_method = aggregation_method
        self.f = f
        self.m = m

    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return None, {}

        client_ndarrays = [fl.common.parameters_to_ndarrays(fit_res.parameters) for _, fit_res in results]
        flat_vectors, shapes_list = [], []
        for nd in client_ndarrays:
            flat, shapes = flatten_weights(nd)
            flat_vectors.append(flat)
            shapes_list.append(shapes)
        V = np.stack(flat_vectors)

        if self.aggregation_method == "fedavg":
            agg_flat = V.mean(axis=0)
        elif self.aggregation_method == "krum":
            agg_flat, _ = krum(V, self.f)
        elif self.aggregation_method == "multikrum":
            agg_flat, _ = multi_krum(V, self.f, m=self.m)
        elif self.aggregation_method == "trimmedmean":
            agg_flat = trimmed_mean(V, self.f)
        else:
            raise ValueError(f"Unknown aggregation_method: {self.aggregation_method}")

        agg_ndarrays = unflatten_weights(agg_flat, shapes_list[0])
        parameters_aggregated = fl.common.ndarrays_to_parameters(agg_ndarrays)
        return parameters_aggregated, {}

print("RobustFedAvg strategy defined.")

**What this cell does:** Runs the identical 8-round, 10-client Flower simulation four times, changing only the strategy's `aggregation_method`. `client_fn` returns a `MaliciousClient` for the first `NUM_MALICIOUS` client IDs and a `FedAvgClient` for the rest. `evaluate_fn`, called by Flower on the server after every round, evaluates the aggregated global model on the **full** MNIST test set and records one accuracy per round — the same pattern m29 uses. All four runs reuse the same `partitions`, so the only thing that changes between curves is the aggregation rule.

In [ ]:
NUM_ROUNDS = 8
LOCAL_EPOCHS = 1
BATCH_SIZE = 32

def client_fn(cid: str) -> fl.client.Client:
    idx = int(cid)
    x_tr, y_tr, x_te, y_te = partitions[idx]
    if idx < NUM_MALICIOUS:
        return MaliciousClient(x_tr, y_tr, x_te, y_te, scale_factor=SCALE_FACTOR).to_client()
    return FedAvgClient(x_tr, y_tr, x_te, y_te).to_client()

def run_robust_simulation(aggregation_method: str, f: int = NUM_MALICIOUS, m: int = None) -> List[float]:
    '''Run one Flower simulation with the given aggregation rule, returning per-round global accuracy.'''
    global_model = create_model()
    accuracies: List[float] = []

    def evaluate_fn(server_round, parameters, config):
        weights = fl.common.parameters_to_ndarrays(parameters) if hasattr(parameters, "tensors") else parameters
        global_model.set_weights(weights)
        loss, accuracy = global_model.evaluate(x_test, y_test, verbose=0)
        if server_round > 0:
            accuracies.append(accuracy)
            print(f"  [{aggregation_method}] round {server_round}: accuracy = {accuracy:.4f}")
        return loss, {"accuracy": accuracy}

    strategy = RobustFedAvg(
        aggregation_method=aggregation_method, f=f, m=m,
        fraction_fit=1.0, fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS, min_evaluate_clients=NUM_CLIENTS, min_available_clients=NUM_CLIENTS,
        evaluate_fn=evaluate_fn,
        on_fit_config_fn=lambda rnd: {"local_epochs": LOCAL_EPOCHS, "batch_size": BATCH_SIZE},
    )

    fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=NUM_CLIENTS,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=strategy,
    )
    return accuracies

print(f"Running 4 aggregation rules under attack (f={NUM_MALICIOUS}, Lambda={SCALE_FACTOR})...\n")
partA_results = {
    "FedAvg (no defense)": run_robust_simulation("fedavg"),
    "Krum (m=1)": run_robust_simulation("krum"),
    f"Multi-Krum (m={NUM_CLIENTS - NUM_MALICIOUS})": run_robust_simulation("multikrum", m=NUM_CLIENTS - NUM_MALICIOUS),
    f"Trimmed Mean (beta={NUM_MALICIOUS})": run_robust_simulation("trimmedmean"),
}

plt.figure(figsize=(9, 6))
for label, acc_curve in partA_results.items():
    plt.plot(range(1, len(acc_curve) + 1), acc_curve, marker='o', label=label)
plt.xlabel('Round')
plt.ylabel('Global Accuracy')
plt.title(f'Byzantine-robust aggregation under the m30 scaling attack (f={NUM_MALICIOUS}, Lambda={SCALE_FACTOR})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nFinal-round accuracy:")
for label, acc_curve in partA_results.items():
    print(f"  {label:28s}: {acc_curve[-1]:.4f}")

Expected output: FedAvg's accuracy should collapse toward random-guess level ($\approx 0.10$) within a few rounds, while Krum, Multi-Krum, and Trimmed Mean should all reach noticeably higher accuracy, with Krum typically a little behind Multi-Krum and Trimmed Mean (it trains the global model using only 1 of the 10 clients' updates per round, instead of $n-f=8$).

**What this cell does:** Sweeps Multi-Krum's $m$ over $\{1, 2, 4, 6, 8, 10\}$ under the identical attack, reusing the $m=1$ (Krum) and $m=8$ (Multi-Krum) results already computed above and running the remaining three values of $m$ as new simulations. Plots final-round accuracy against $m$ to chart the performance-robustness trade-off: accuracy should hold up while $m \leq n-f=8$ and then fall once $m$ is forced to include at least one malicious client's update.

In [ ]:
m_values = [1, 2, 4, 6, 8, 10]
m_final_accuracy = {
    1: partA_results["Krum (m=1)"][-1],
    NUM_CLIENTS - NUM_MALICIOUS: partA_results[f"Multi-Krum (m={NUM_CLIENTS - NUM_MALICIOUS})"][-1],
}
for m in m_values:
    if m not in m_final_accuracy:
        print(f"Running Multi-Krum with m={m}...")
        acc_curve = run_robust_simulation("multikrum", m=m)
        m_final_accuracy[m] = acc_curve[-1]

m_sweep_accuracies = [m_final_accuracy[m] for m in m_values]

plt.figure(figsize=(8, 5))
plt.plot(m_values, m_sweep_accuracies, marker='s', color='tab:blue')
plt.axvline(NUM_CLIENTS - NUM_MALICIOUS, color='gray', linestyle='--', label=f'm = n - f = {NUM_CLIENTS - NUM_MALICIOUS}')
plt.xlabel('m (number of client updates averaged by Multi-Krum)')
plt.ylabel(f'Final-round accuracy (round {NUM_ROUNDS})')
plt.title('Multi-Krum performance-robustness trade-off under attack')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Final accuracy by m:", {m: round(acc, 4) for m, acc in m_final_accuracy.items()})

Expected output: accuracy should stay comparably high for $m \in \{1, 2, 4, 6, 8\}$ and then drop noticeably at $m=10$, the point at which Multi-Krum has no choice but to average in at least one of the two malicious clients' updates alongside the honest ones.

## Part B: Secure Aggregation — Double-Masking Demonstration

SecAgg is a cryptographic protocol, not a training algorithm: its masking and cancellation arithmetic works identically no matter what the masked vectors represent, so this demonstration uses small toy vectors for clarity rather than full Keras weight vectors — the formulas below would apply exactly as written to real client updates like the ones Parts A and C work with.

**Recap of the formula.** Each toy client $u$ sends $y_u = x_u + b_u + \sum_{v>u} s_{u,v} - \sum_{v<u} s_{u,v}$, where $b_u$ is $u$'s own random self-mask and $s_{u,v}=s_{v,u}$ is a secret shared only between $u$ and $v$. When every client submits, summing all $y_u$ makes every pairwise term cancel automatically, leaving $\sum_u y_u = \sum_u x_u + \sum_u b_u$ — the self-masks are the only thing left for the server to remove.

**What this cell does:** Builds 5 toy clients, each with a secret contribution `x_u`, a random self-mask `b_u`, and symmetric pairwise masks `s[u, v] == s[v, u]`. It computes each client's masked message `y_u`, sums all five, and checks the two claims from the recap: the sum equals `sum(x) + sum(b)` exactly (pairwise masks cancel on their own), and that sum is *not* equal to `sum(x)` alone (the self-masks genuinely remain, and must be removed by some other mechanism — demonstrated in the next cell).

In [ ]:
N_TOY = 5
DIM_TOY = 4

x = {u: np.random.randn(DIM_TOY) for u in range(N_TOY)}        # each client's true secret contribution
b = {u: np.random.randn(DIM_TOY) for u in range(N_TOY)}        # each client's self-mask

s = {}                                                          # symmetric pairwise masks: s[u,v] == s[v,u]
for u in range(N_TOY):
    for v in range(u + 1, N_TOY):
        shared = np.random.randn(DIM_TOY)
        s[(u, v)] = shared
        s[(v, u)] = shared

def masked_message(u):
    y = x[u] + b[u]
    for v in range(N_TOY):
        if v == u:
            continue
        y = y + s[(u, v)] if v > u else y - s[(u, v)]
    return y

y = {u: masked_message(u) for u in range(N_TOY)}
server_sum = sum(y.values())

sum_x = sum(x.values())
sum_b = sum(b.values())

print("Does server_sum == sum(x) + sum(b)?  ", np.allclose(server_sum, sum_x + sum_b))
print("Does server_sum == sum(x) alone?     ", np.allclose(server_sum, sum_x))
print("\n-> Pairwise masks cancel for free. Self-masks remain and must be removed separately")
print("   (via Shamir secret sharing, so no single party -- not even the server -- can remove")
print("   them without cooperation from a threshold number of the other clients).")

**What this cell does:** Implements a small Shamir $(t, n)$ secret-sharing scheme over real numbers (production SecAgg uses a finite field for information-theoretic security; this version is for intuition only, as noted in the lecture). `make_shares` builds a random degree-$(t-1)$ polynomial whose constant term is the secret, and evaluates it at $n$ points to produce $n$ shares. `reconstruct` uses Lagrange interpolation on any $t$ of those shares to recover the polynomial's constant term exactly. The cell then shows that $t-1$ shares are consistent with *any* candidate secret.

In [ ]:
def make_shares(secret, t, n, seed=0):
    '''Random degree-(t-1) polynomial with constant term `secret`; returns n (x, y) shares.'''
    rng = np.random.RandomState(seed)
    coeffs = [secret] + list(rng.randn(t - 1))   # coeffs[0] is the secret
    xs = np.arange(1, n + 1)
    ys = np.array([sum(c * (x_val ** i) for i, c in enumerate(coeffs)) for x_val in xs])
    return list(zip(xs, ys))

def reconstruct(shares):
    '''Lagrange interpolation at x=0 to recover the polynomial's constant term (the secret).'''
    total = 0.0
    for i, (xi, yi) in enumerate(shares):
        term = yi
        for j, (xj, _) in enumerate(shares):
            if i != j:
                term *= (0 - xj) / (xi - xj)
        total += term
    return total

T, N_SHARES = 3, 5
SECRET = 42.0
shares = make_shares(SECRET, T, N_SHARES, seed=1)
print("All shares:", [(x_, round(y_, 3)) for x_, y_ in shares])

recovered = reconstruct(shares[:T])
print(f"\nReconstructing from {T} shares: {recovered:.6f}  (true secret: {SECRET})")
assert abs(recovered - SECRET) < 1e-6

# With only t-1 = 2 shares, show that a DIFFERENT candidate secret is equally consistent.
partial_shares = shares[:T - 1]
candidate_secret = SECRET + 1000.0
(x1, y1), (x2, y2) = partial_shares
fake_poly_points = [(0, candidate_secret), (x1, y1), (x2, y2)]
fake_recovered = reconstruct(fake_poly_points)
print(f"\nWith only {T-1} shares, candidate secret {candidate_secret} fits a valid")
print(f"degree-{T-1} polynomial through those points just as well: reconstruct(...) = {fake_recovered:.3f}")
print("-> t-1 shares reveal nothing: any candidate secret is consistent with some valid polynomial.")

Expected output: reconstructing from $t=3$ shares recovers `42.0` exactly. The final check confirms that with only $t-1=2$ shares, an arbitrary, very different candidate secret can be made to fit a valid polynomial through those same points just as well.

## Part C: Local Differential Privacy as a Flower Client

**Recap of the formula.** Each client clips its weight *delta* to L2 norm $C$, then adds noise $\mathcal{N}(0,\sigma^2 I)$ with $\sigma = C\sqrt{2\ln(1.25/\delta)}/\epsilon$ (the classical Gaussian mechanism, valid for $\epsilon \in (0,1)$ and commonly used somewhat more broadly in practice), before returning it to the server. Smaller $\epsilon$ (stronger privacy) means larger $\sigma$ (more noise).

**What this cell does:** Defines `PrivateClient(FedAvgClient)`, which overrides `fit` to clip and noise its weight delta using the same `flatten_weights`/`unflatten_weights` helpers from Part A, exactly as `MaliciousClient` overrides `fit` to scale its delta. `epsilon=None` means no DP noise at all (the baseline). Printing $\sigma$ for five values of $\epsilon$ confirms $\sigma \propto 1/\epsilon$.

In [ ]:
CLIP_NORM = 1.5
DELTA = 1e-5

class PrivateClient(FedAvgClient):
    '''Clips its weight delta to L2 norm clip_norm, then adds Gaussian noise calibrated to (eps, delta)-LDP.'''
    def __init__(self, x_train, y_train, x_test, y_test, epsilon=None, clip_norm=CLIP_NORM, delta=DELTA):
        super().__init__(x_train, y_train, x_test, y_test)
        self.epsilon = epsilon
        self.clip_norm = clip_norm
        self.delta = delta

    def fit(self, parameters, config):
        initial_weights = [w.copy() for w in parameters]
        self.model.set_weights(parameters)
        local_epochs = config.get('local_epochs', 1)
        batch_size = config.get('batch_size', 32)
        self.model.fit(self.x_train, self.y_train, epochs=local_epochs, batch_size=batch_size, verbose=0)
        updated_weights = self.model.get_weights()

        deltas = [upd - init for upd, init in zip(updated_weights, initial_weights)]
        flat, shapes = flatten_weights(deltas)
        norm = np.linalg.norm(flat)
        flat = flat * min(1.0, self.clip_norm / (norm + 1e-12))
        if self.epsilon is not None:
            sigma = self.clip_norm * np.sqrt(2 * np.log(1.25 / self.delta)) / self.epsilon
            flat = flat + np.random.normal(0, sigma, size=flat.shape)
        noisy_deltas = unflatten_weights(flat, shapes)

        noisy_weights = [init + d for init, d in zip(initial_weights, noisy_deltas)]
        return noisy_weights, len(self.x_train), {}

print(f"clip_norm={CLIP_NORM}, delta={DELTA}\n")
print(f"{'epsilon':>8} | {'sigma':>8}")
for eps in [0.5, 1.0, 2.0, 5.0, 10.0]:
    sigma = CLIP_NORM * np.sqrt(2 * np.log(1.25 / DELTA)) / eps
    print(f"{eps:>8.1f} | {sigma:>8.3f}")
print("\nsigma is exactly proportional to 1/epsilon, as the formula says.")

**What this cell does:** Runs the identical 8-round, 10-client, all-honest Flower simulation (plain `fl.server.strategy.FedAvg`, no robust aggregation needed since there is no attacker in this part) once per $\epsilon$ in $\{0.5, 1, 2, 5, 10\}$ plus a no-DP baseline, with every client a `PrivateClient` using that $\epsilon$. Plots final-round accuracy against $\epsilon$ — the accuracy cost of the privacy budget.

In [ ]:
def run_private_simulation(epsilon):
    '''Run one all-honest Flower simulation with every client using PrivateClient at this epsilon.'''
    def client_fn_private(cid):
        idx = int(cid)
        x_tr, y_tr, x_te, y_te = partitions[idx]
        return PrivateClient(x_tr, y_tr, x_te, y_te, epsilon=epsilon).to_client()

    global_model = create_model()
    accuracies = []

    def evaluate_fn(server_round, parameters, config):
        weights = fl.common.parameters_to_ndarrays(parameters) if hasattr(parameters, "tensors") else parameters
        global_model.set_weights(weights)
        loss, accuracy = global_model.evaluate(x_test, y_test, verbose=0)
        if server_round > 0:
            accuracies.append(accuracy)
        return loss, {"accuracy": accuracy}

    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0, fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS, min_evaluate_clients=NUM_CLIENTS, min_available_clients=NUM_CLIENTS,
        evaluate_fn=evaluate_fn,
        on_fit_config_fn=lambda rnd: {"local_epochs": LOCAL_EPOCHS, "batch_size": BATCH_SIZE},
    )
    fl.simulation.start_simulation(
        client_fn=client_fn_private,
        num_clients=NUM_CLIENTS,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=strategy,
    )
    return accuracies

eps_values = [0.5, 1.0, 2.0, 5.0, 10.0]
ldp_results = {}
for eps in eps_values:
    print(f"Running LDP simulation at epsilon={eps}...")
    ldp_results[eps] = run_private_simulation(eps)[-1]
print("Running no-DP baseline...")
no_dp_final = run_private_simulation(epsilon=None)[-1]

plt.figure(figsize=(8, 5))
plt.plot(eps_values, [ldp_results[e] for e in eps_values], marker='o', label='With LDP (Gaussian mechanism)')
plt.axhline(no_dp_final, color='gray', linestyle='--', label='No DP (baseline)')
plt.xlabel('Privacy budget epsilon (smaller = more private)')
plt.ylabel(f'Final-round accuracy (round {NUM_ROUNDS})')
plt.title('Accuracy cost of Local Differential Privacy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nFinal accuracy by epsilon:")
for eps in eps_values:
    print(f"  epsilon={eps:>4.1f}: {ldp_results[eps]:.4f}")
print(f"  no DP     : {no_dp_final:.4f}")

Expected output: accuracy should increase (closer to the no-DP baseline) as $\epsilon$ grows from 0.5 to 10, with $\epsilon=0.5$ noticeably worse than the no-DP run and $\epsilon=10$ closer to it — the accuracy cost of privacy, directly as a function of the privacy budget.

## Comprehension Questions

1. Krum's Byzantine fault tolerance guarantee requires $n \geq 2f+3$. What could go wrong if this lab used $n=10$ clients with $f=4$ malicious ones instead of $f=2$?
2. In Part A, Krum ($m=1$) typically ends with lower accuracy than Multi-Krum ($m=n-f=8$), even though both successfully resist the attack. Why does keeping only one client's update per round cost accuracy, and why does Multi-Krum's larger $m$ not reopen the same vulnerability that FedAvg's $m=n$ has?
3. `RobustFedAvg.aggregate_fit` flattens every client's weights into one vector before scoring them. Why would Krum's distance-based scoring not make sense if it were instead applied layer by layer, scoring each layer's weights completely independently of the others?
4. In Part B, the server only ever sees `server_sum`, `sum(x) + sum(b)` for all five clients combined. Name one specific thing about any individual client's `x_u` that this protocol guarantees the server can never learn, even in principle, from that sum alone.
5. In Part C, why does `PrivateClient` clip and noise the weight *delta* (the change from the received global weights) rather than the raw updated weights themselves?
6. Suppose a deployment wanted to run `RobustFedAvg` and SecAgg together exactly as implemented in this lab. Explain concretely why `aggregate_fit` could not run on the masked values a real SecAgg server would receive.

**Answer 1:** With $f=4$ Byzantine clients among $n=10$, the condition $n \geq 2f+3=11$ fails. The four malicious clients could coordinate so each sits among the others' nearest neighbors, manufacturing artificially low Krum scores for themselves. Krum's guarantee no longer applies once this is possible, and the lowest-scoring vector, the one `krum_scores` would select, could end up being a malicious one instead of an honest one.

**Answer 2:** Krum's `aggregate_fit` call discards 9 of the 10 clients' updates every round and keeps only 1, so the global model trains on whatever that single client's local data happened to produce that round, with none of the variance reduction that averaging multiple clients' updates would normally provide. Multi-Krum with $m=8=n-f$ averages 8 updates instead of 1, using far more of the clients' combined training signal every round, without reopening FedAvg's vulnerability, because the 8 selected updates are chosen by the same distance-based scoring that (under $n\geq2f+3$) keeps the 2 malicious updates out of the selected set in the first place.

**Answer 3:** Krum's score compares *whole client updates* to find the ones that look mutually consistent overall. If scored layer by layer instead, a malicious client could submit an update that looks consistent with the honest cluster on most layers but is wildly scaled on just one layer, and a per-layer score might never flag that layer as an outlier if the other layers from that same client score well; worse, nothing would stop the aggregated result from combining "safe" layers from one client's scoring with a layer from another client in a way that doesn't correspond to any client's actual coherent update, except by coincidence. Flattening into one vector makes the scoring judge each client's update as a single, indivisible contribution, matching how `MaliciousClient.fit` actually attacks: by scaling the *entire* delta at once.

**Answer 4:** The server can never learn any individual client's raw contribution $x_u$ by itself — not its value, not even which of the five clients contributed which part of the sum. `server_sum` (even after the self-masks are eventually removed through secret sharing) is one single aggregate number for all five clients together; there is no operation available to the server that isolates one client's term from that sum.

**Answer 5:** The delta is what the client is actually contributing to the federated update; clipping and noising it bounds and protects exactly the quantity this client's local training changed, regardless of how large the (shared, public) starting global weights happen to be. Clipping the raw updated weights instead would also bound the already-public global weights' own magnitude, wasting most of the clip budget $C$ on information that was never private in the first place and was already known to the server before this round even started.

**Answer 6:** `aggregate_fit` calls `fl.common.parameters_to_ndarrays(fit_res.parameters)` on each individual client's result to get that client's actual weight values, then flattens and scores them with `krum_scores`. Under SecAgg, the server never receives any individual client's `Parameters` object at all — it only ever receives one combined masked sum across all clients. There is no `results` list of individual `FitRes` for `aggregate_fit` to loop over in the way this cell's code requires; SecAgg's entire design goal was to make sure that list never reaches the server in the first place.

## Verification

**What this cell does:** Three checks. Verification 1 re-confirms the `multi_krum` special cases and the $n\geq2f+3$ bound for this lab's settings. Verification 2 checks that under the m30 attack, all three robust methods end Part A's 8-round run with higher accuracy than plain FedAvg, and that the $m$-sweep's accuracy is at least as high at $m=n-f=8$ as it is at $m=10$. Verification 3 checks the SecAgg cancellation identity and the LDP $\sigma \propto 1/\epsilon$ relationship. All three must print PASS.

In [ ]:
# Verification 1: special cases and the Byzantine-tolerance bound.
assert np.allclose(multi_krum(_V_test, NUM_MALICIOUS, m=1)[0], krum(_V_test, NUM_MALICIOUS)[0])
assert np.allclose(multi_krum(_V_test, NUM_MALICIOUS, m=NUM_CLIENTS)[0], _V_test.mean(axis=0))
assert NUM_CLIENTS >= 2 * NUM_MALICIOUS + 3, "This lab's (NUM_CLIENTS, NUM_MALICIOUS) must satisfy n >= 2f+3."
print("PASS: multi_krum special cases and the n >= 2f+3 bound hold for this lab's settings.")

# Verification 2: robust methods beat FedAvg under attack; m-sweep holds up to m = n - f.
fedavg_final = partA_results["FedAvg (no defense)"][-1]
for label in ["Krum (m=1)", f"Multi-Krum (m={NUM_CLIENTS-NUM_MALICIOUS})", f"Trimmed Mean (beta={NUM_MALICIOUS})"]:
    assert partA_results[label][-1] > fedavg_final, f"{label} should end with higher accuracy than FedAvg."
assert m_final_accuracy[NUM_CLIENTS - NUM_MALICIOUS] >= m_final_accuracy[NUM_CLIENTS], \
    "Accuracy at m = n - f should be at least as high as at m = n under attack."
print("PASS: all three robust methods beat FedAvg under attack, and m = n - f outperforms m = n.")

# Verification 3: SecAgg cancellation identity; LDP sigma trend.
assert np.allclose(server_sum, sum_x + sum_b), "Pairwise masks must cancel exactly."
assert not np.allclose(server_sum, sum_x), "Self-masks must NOT cancel on their own."
sigmas = [CLIP_NORM * np.sqrt(2 * np.log(1.25 / DELTA)) / e for e in eps_values]
assert all(sigmas[i] > sigmas[i + 1] for i in range(len(sigmas) - 1)), "sigma must decrease as epsilon grows."
print("PASS: SecAgg cancellation identity holds and sigma decreases as epsilon grows.")

## Optional Challenges

1. Push past the guarantee: set `NUM_MALICIOUS = 4` (so `n >= 2f+3` fails for `n=10`) and re-run the Part A comparison. Does Krum still protect accuracy?
2. Implement coordinate-wise **median** (`np.median(V, axis=0)`) as a fifth `aggregation_method` in `RobustFedAvg`, add it to the Part A comparison plot, and compare it to Trimmed Mean.
3. Extend the Part B demonstration to simulate one client dropping out after sharing its secrets but before submitting `y_u`. Show what leftover pairwise terms the other clients' submissions now contain.
4. Compose Part A and Part C: make every honest client in the Part A attack scenario a `PrivateClient` instead of a plain `FedAvgClient` (keeping the malicious clients as-is), and re-run Multi-Krum. Does adding LDP noise to the honest clients change Multi-Krum's final accuracy compared to the un-noised Part A run with the same attack?

## Assignment:

Extend the experiments in this notebook by modifying the settings below. Complete all four tasks, fill in the results table, and answer the questions below.

**Baseline (what the notebook runs above):** `NUM_CLIENTS=10`, `NUM_MALICIOUS=2`, `SCALE_FACTOR=50`, `NUM_ROUNDS=8` for Part A; `m_values=[1,2,4,6,8,10]` for the $m$-sweep; `eps_values=[0.5,1,2,5,10]`, `CLIP_NORM=1.5`, `DELTA=1e-5` for Part C.

Tip: restore each baseline value before moving to the next task.

#### Task 1 — Stronger attack ($\Lambda$: 50 → 200)
Restore `NUM_MALICIOUS = 2`. Set `SCALE_FACTOR = 200` and re-run the Part A four-way comparison.

*Question:* Does FedAvg's final accuracy drop further than at $\Lambda=50$? Do Krum, Multi-Krum, and Trimmed Mean's final accuracies change much at all? Explain using the fact that none of `krum_scores`, `multi_krum`, or `trimmed_mean` reference $\Lambda$ anywhere in their formulas.

#### Task 2 — More malicious clients, still within the guarantee ($f$: 2 → 3)
Restore `SCALE_FACTOR = 50`. Set `NUM_MALICIOUS = 3` (still satisfies $n=10 \geq 2(3)+3=9$) and re-run the Part A comparison, remembering that Multi-Krum's default `m` is `NUM_CLIENTS - NUM_MALICIOUS`.

*Question:* How does Krum's final accuracy at $f=3$ compare to $f=2$? Relate your answer to how many of the $n-f-2$ nearest-neighbor slots in the score formula are now available to honest vectors.

#### Task 3 — Smaller clipping bound ($C$: 1.5 → 0.5)
Restore `NUM_MALICIOUS = 2`. Set `CLIP_NORM = 0.5` in Part C and re-run the $\epsilon$ sweep.

*Question:* How does the whole accuracy-vs-$\epsilon$ curve shift compared to the baseline? Using $\sigma = C\sqrt{2\ln(1.25/\delta)}/\epsilon$, explain why a smaller clip bound changes the noise added at every $\epsilon$, not just the clipping itself.

#### Task 4 — More local training per round ($E$: 1 → 3)
Restore `CLIP_NORM = 1.5`. Set `LOCAL_EPOCHS = 3` and re-run both the Part A comparison and the Part C $\epsilon$ sweep.

*Question:* Does more local training per round help or hurt FedAvg under attack? Does it help or hurt the robust methods? Relate your answer to how much a malicious client's locally trained delta changes before `MaliciousClient.fit` scales it.

#### Results table (fill in the final-round accuracies)

| Configuration | FedAvg | Krum | Multi-Krum | Trimmed Mean |
|---|---|---|---|---|
| Baseline ($\Lambda$=50, $f$=2) | | | | |
| Task 1 ($\Lambda$=200) | | | | |
| Task 2 ($f$=3) | | | | — |
| Task 4 ($E$=3) | | | | |

| Configuration | $\epsilon$=0.5 final acc | $\epsilon$=10 final acc |
|---|---|---|
| Baseline ($C$=1.5) | | |
| Task 3 ($C$=0.5) | | |
| Task 4 ($E$=3) | | |

#### Written questions

1. FedAvg's final accuracy collapses further as $\Lambda$ grows; the three robust methods' do not change much. State in one sentence why none of `krum_scores`, `multi_krum`, or `trimmed_mean` contains $\Lambda$ anywhere in its formula, and what that implies about how large an attacker's scale factor could be before these defenses would need to change.
2. A deployment is choosing between Krum ($m=1$), Multi-Krum with $m=n-f$, and Trimmed Mean with $\beta=f$. Given your Task 2 results, rank the three by how well they tolerate $f$ approaching its maximum allowed value, and explain the ranking using each rule's formula.
3. Revisit the combined-defense reasoning from the lecture and notes.md: name one pair of the three defenses in this lab (robust aggregation, SecAgg, LDP) that composes cleanly, and one pair that is in tension, and justify each using what you implemented in Parts A-C.

**Deliverables:** this notebook with all cells executed top to bottom, the completed results tables, and your answers to the four task questions plus the three written questions above.